# WorkIQ SharePoint Agent

Foundry에 **WorkIQ SharePoint** 에이전트를 등록하고, SDK에서 호출하는 노트북입니다.

**사전 준비**
1. **WorkIQ SharePoint connection** 등록 — Foundry portal의 *Build > Tools*에서
2. Foundry 프로젝트에 모델 배포 (예: `gpt-4.1`)
3. `.env.example`을 복사해 `.env` 작성
   ```bash
   cp .env.example .env
   ```
4. Azure 로그인 (`az login`)

## 1. 패키지 설치

In [ ]:
%pip install --quiet --pre "azure-ai-projects>=2.0.0" azure-identity python-dotenv

## 2. 환경 변수 로드

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_AI_PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
WORKIQ_SHAREPOINT_CONNECTION_NAME = os.environ["WORKIQ_SHAREPOINT_CONNECTION_NAME"]
MODEL_DEPLOYMENT_NAME = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-4.1")
AGENT_NAME = os.environ.get("AGENT_NAME", "workiq-sharepoint-agent")
AZURE_TENANT_ID = os.environ.get("AZURE_TENANT_ID", "")


print(f"Project endpoint   : {AZURE_AI_PROJECT_ENDPOINT}")
print(f"WorkIQ SharePoint  : {WORKIQ_SHAREPOINT_CONNECTION_NAME}")
print(f"Model deployment   : {MODEL_DEPLOYMENT_NAME}")
print(f"Agent name         : {AGENT_NAME}")

## 3. Azure 로그인

`DefaultAzureCredential`을 사용합니다. 사전에 `az login`을 실행해 주세요.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
print("DefaultAzureCredential 사용")

project_client = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=credential,
)

## 4. Foundry 에이전트 등록 (포탈에 표시)

`project_client.agents.create_version()`을 사용하여 Foundry 포탈에 표시되는 에이전트를 등록합니다.

In [ ]:
from azure.ai.projects.models import (
    PromptAgentDefinition,
    MCPTool,
)

definition = PromptAgentDefinition(
    model=MODEL_DEPLOYMENT_NAME,
    instructions=(
        "당신은 사내 **WorkIQ SharePoint** 자료를 검색하여 답변하는 한국어 어시스턴트입니다. "
        "질문에 답할 때는 WorkIQ SharePoint 도구를 호출하여 근거 문서를 찾고, "
        "답변에는 인용한 문서의 제목과 링크를 함께 제시하세요. "
        "WorkIQ SharePoint에서 근거를 찾을 수 없으면 '확인되지 않습니다'라고 솔직히 답하세요."
    ),
    tools=[
        MCPTool(
            server_label=WORKIQ_SHAREPOINT_CONNECTION_NAME,
            server_url="https://agent365.svc.cloud.microsoft/agents/servers/mcp_SharePointRemoteServer",
            require_approval="never",
            project_connection_id=WORKIQ_SHAREPOINT_CONNECTION_NAME,
        )
    ],
)

agent_version = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=definition,
    description="사내 WorkIQ SharePoint 문서 검색 에이전트",
)
print(f"Agent registered — name: {agent_version.name}, version: {agent_version.version}, id: {agent_version.id}")

## 5. 테스트 유저 로그인

다른 유저로 에이전트를 호출해보려면 `USE_BROWSER_LOGIN = True`로 설정하세요.  
브라우저 로그인 창이 열리며, `.env`의 `AZURE_TENANT_ID`로 테넌트를 지정할 수 있습니다.

In [ ]:
from azure.identity import InteractiveBrowserCredential

USE_BROWSER_LOGIN = True

if USE_BROWSER_LOGIN:
    test_credential = InteractiveBrowserCredential(
        tenant_id=AZURE_TENANT_ID or None,
    )
    test_credential.get_token("https://management.azure.com/.default")
    print("브라우저 로그인 성공! (테스트 유저)")
else:
    test_credential = credential
    print("기본 credential 사용 (에이전트 생성자와 동일)")

## 6. 에이전트 호출

테스트 유저의 credential로 Foundry 에이전트를 호출합니다.

In [ ]:
from azure.ai.projects import AIProjectClient

test_project_client = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=test_credential,
)
foundry_client = test_project_client.get_openai_client()

user_question = "Kichul Private Site 에 있는 모든 파일들 리스트업해줘"

response = foundry_client.responses.create(
    input=[{"role": "user", "content": user_question}],
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "version": str(agent_version.version),
            "type": "agent_reference",
        }
    },
)
print(response.output_text)

## 7. 후속 질문

In [ ]:
response = foundry_client.responses.create(
    input=[{"role": "user", "content": "방금 답변한 문서에서 TBB 에 대한 정보 요약해서 알려줘"}],
    extra_body={
        "agent_reference": {
            "name": AGENT_NAME,
            "version": str(agent_version.version),
            "type": "agent_reference",
        }
    },
    previous_response_id=response.id,
)
print(response.output_text)

## 8. 정리 (선택)

테스트가 끝나면 에이전트를 삭제할 수 있습니다. 포탈에서도 제거됩니다.

In [ ]:
# project_client.agents.delete(agent_name=AGENT_NAME)
# print(f"Deleted agent: {AGENT_NAME}")